# Threat Actor Analysis API — Gemini Colab prototype

**Project:** `mitre-attack-threat-group-similarity`  
**Homework:** Gilusis mokymas ND1, part 1.2.2  
**Problem:** given a list of observed [MITRE ATT&CK](https://attack.mitre.org/) technique IDs, retrieve similar historical incidents and estimate which threat **groups** are most consistent with that evidence.

This notebook is a working prototype of the **Threat Actor Analysis API** in the repository README. Each numbered section is one UML component (plus setup and an end-to-end run):

| Section | UML component | What it does |
| --- | --- | --- |
| 2 | GitHub Repository → Knowledge Base | Load `data/incidents_strict.json` from the repo's raw GitHub URL |
| 3 | Query Processor | Validate / normalize / dedupe ATT&CK IDs |
| 4 | Vector Index | Gemini embeddings in FAISS (or TF-IDF/Jaccard fallback) |
| 5 | RAG Retriever | Top-k similar incidents with group, techniques, description |
| 6 | Gemini API | `prompts/prompt.json` → structured JSON in the `y` shape |
| 7 | Result Generator | Parse/validate JSON, bar chart, matching vs missing table |
| 8 | REST API `POST /analyze` | `analyze(techniques)` (+ optional in-notebook FastAPI demo) |
| 9 | End-to-end | The 3 `X` inputs from `data/examples.json` |

**Run in Colab:** Runtime → Run all. Add a Colab Secret named `GEMINI_API_KEY` (🔑 Secrets panel) to enable Gemini embeddings and generation. **Without a key the notebook still runs end-to-end** using the TF-IDF/Jaccard fallback; Gemini steps are skipped and labelled clearly.

Do not paste an API key into a cell. Keys must never be committed.


## 1. System overview (README architecture)

External systems (for example a SIEM) send MITRE technique IDs to `POST /analyze`. The API validates the query, retrieves similar historical incidents from a vector index over the knowledge base, asks Gemini to attribute the top 3 groups **using only that retrieved evidence**, then returns structured JSON.

The knowledge base itself is the GitHub file `data/incidents_strict.json` (662 incidents derived from MITRE ATT&CK: `incident_id`, `group`, `techniques`, `description`).


## 2. Setup — packages, `GEMINI_API_KEY`, Knowledge Base

Install the Google Gen AI SDK (`google-genai`) plus FAISS (or numpy) for the vector index. Then read **`GEMINI_API_KEY` from Colab Secrets** (`google.colab.userdata`), with a graceful fallback when the secret is missing so the rest of the notebook still runs.

This matches the official Gemini Colab pattern: never hard-code the key; store it in the 🔑 Secrets panel as `GEMINI_API_KEY`.


In [ ]:
import subprocess
import sys


def install_packages(packages):
    """Use %pip in IPython/Colab; fall back to python -m pip otherwise."""
    try:
        ipy = get_ipython()  # type: ignore[name-defined]
    except NameError:
        ipy = None
    if ipy is not None:
        ipy.run_line_magic("pip", "install -qU " + " ".join(packages))
        return
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *packages])


install_packages(
    [
        "google-genai>=1.0.0",
        "numpy",
        "scikit-learn",
        "pandas",
        "matplotlib",
        "fastapi",
        "uvicorn",
        "requests",
    ]
)
try:
    install_packages(["faiss-cpu"])
except Exception as exc:
    print("faiss-cpu could not be installed; numpy similarity will be used.", exc)

print("Package install step finished.")


### Read `GEMINI_API_KEY` (Colab Secret) without crashing

In Colab: Secrets → add `GEMINI_API_KEY` → enable notebook access.  
Outside Colab: the same name is read from the environment. If the key is absent, `HAS_GEMINI` is `False` and later Gemini cells **skip** instead of failing.


In [ ]:
import os

GEMINI_API_KEY = None
IN_COLAB = False

try:
    from google.colab import userdata

    IN_COLAB = True
    try:
        GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
    except userdata.SecretNotFoundError:
        print("Colab Secret GEMINI_API_KEY was not found.")
        GEMINI_API_KEY = None
    except Exception as exc:
        print(f"Could not read Colab Secret GEMINI_API_KEY ({type(exc).__name__}: {exc}).")
        GEMINI_API_KEY = None
except ImportError:
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
    if not GEMINI_API_KEY:
        print("Not running in Colab and no GEMINI_API_KEY / GOOGLE_API_KEY env var.")

if isinstance(GEMINI_API_KEY, str):
    GEMINI_API_KEY = GEMINI_API_KEY.strip() or None

HAS_GEMINI = bool(GEMINI_API_KEY)
client = None

if HAS_GEMINI:
    from google import genai

    client = genai.Client(api_key=GEMINI_API_KEY)
    print("Gemini client ready (key loaded, value not printed).")
else:
    print("HAS_GEMINI = False — embeddings and generation will use the offline fallback.")

# Headless matplotlib when there is no display (local CI / this smoke run).
if not IN_COLAB and not os.environ.get("DISPLAY"):
    import matplotlib

    matplotlib.use("Agg")
    print("Matplotlib backend: Agg")


### Load the Knowledge Base from GitHub (UML: GitHub Repository → DB)

The dataset is `data/incidents_strict.json` on branch `main` of this repo. We also load `prompts/prompt.json` (Gemini instructions) and `data/examples.json` (three homework inputs + the `y` JSON shape).

Shapes:

- **incident:** `incident_id`, `group`, `techniques`, `description`
- **prompt:** `system_prompt`, `user_prompt_template` with `{{techniques}}` and `{{incidents}}`
- **example:** `{ "X": { "techniques": [...] }, "y": { "groups": [...] } }`


In [ ]:
import json
import urllib.request
from pathlib import Path

REPO_OWNER = "MadAcolyte"
REPO_NAME = "mitre-attack-threat-group-similarity"
REPO_BRANCH = "main"
RAW_BASE = f"https://raw.githubusercontent.com/{REPO_OWNER}/{REPO_NAME}/{REPO_BRANCH}"

INCIDENTS_PATH = "data/incidents_strict.json"
PROMPT_PATH = "prompts/prompt.json"
EXAMPLES_PATH = "data/examples.json"


def load_repo_json(relative_path: str):
    """Fetch JSON from the GitHub raw URL; use a local checkout if the network fails."""
    url = f"{RAW_BASE}/{relative_path}"
    request = urllib.request.Request(
        url,
        headers={"User-Agent": "threat-group-similarity-colab"},
    )
    try:
        with urllib.request.urlopen(request, timeout=60) as response:
            payload = json.loads(response.read().decode("utf-8"))
        print(f"Loaded {relative_path} from {url}")
        return payload
    except Exception as exc:
        local = Path(relative_path)
        if local.is_file():
            print(f"GitHub fetch failed for {relative_path} ({exc}); using local {local}.")
            return json.loads(local.read_text(encoding="utf-8"))
        raise RuntimeError(f"Could not load {relative_path} from GitHub or disk") from exc


incidents = load_repo_json(INCIDENTS_PATH)
prompt_spec = load_repo_json(PROMPT_PATH)
examples = load_repo_json(EXAMPLES_PATH)

assert isinstance(incidents, list) and incidents, "incidents_strict.json must be a non-empty list"
for key in ("incident_id", "group", "techniques", "description"):
    assert key in incidents[0], f"incident missing {key}"

print(f"Knowledge base incidents: {len(incidents)}")
print(f"Prompt name={prompt_spec.get('name')} version={prompt_spec.get('version')}")
print(f"Homework examples: {len(examples)}")
print("Sample incident:")
print(json.dumps(incidents[0], indent=2)[:800], "...")


## 3. Query Processor

The Query Processor is the first stage after `POST /analyze`. It:

1. **Validates** each token against `T\d{4}(\.\d{3})?` (for example `T1059` or `T1059.001`)
2. **Normalizes** whitespace and casing (`t1059.001` → `T1059.001`)
3. **Dedupes** while preserving order
4. **Warns** on IDs that are syntactically valid but **unknown to this dataset** (not present in any `incidents_strict.json` incident)

Invalid IDs are dropped, not sent to retrieval.


In [ ]:
import re
from typing import Iterable

TECHNIQUE_RE = re.compile(r"^T\d{4}(\.\d{3})?$", re.IGNORECASE)

KNOWN_TECHNIQUES = {tid for incident in incidents for tid in incident["techniques"]}
print(f"Unique ATT&CK technique IDs in the knowledge base: {len(KNOWN_TECHNIQUES)}")


def normalize_technique_id(raw: str) -> str | None:
    if raw is None:
        return None
    token = str(raw).strip().upper()
    if not token:
        return None
    if TECHNIQUE_RE.fullmatch(token):
        return token
    return None


def process_query(raw_techniques: Iterable[str], known_ids: set[str] | None = None) -> dict:
    """Validate, normalize, dedupe, and flag unknown ATT&CK IDs."""
    known = known_ids if known_ids is not None else KNOWN_TECHNIQUES
    raw_list = list(raw_techniques)
    techniques: list[str] = []
    invalid: list[str] = []
    unknown: list[str] = []
    seen: set[str] = set()

    for raw in raw_list:
        normalized = normalize_technique_id(raw)
        if normalized is None:
            invalid.append(str(raw))
            continue
        if normalized in seen:
            continue
        seen.add(normalized)
        techniques.append(normalized)
        if normalized not in known:
            unknown.append(normalized)

    if invalid:
        print("WARNING: dropped invalid technique IDs:", invalid)
    if unknown:
        print("WARNING: technique IDs not present in incidents_strict.json:", unknown)
    if not techniques:
        print("WARNING: query has no valid technique IDs after validation.")

    return {
        "techniques": techniques,
        "invalid": invalid,
        "unknown": unknown,
        "dropped_duplicates": max(0, len(raw_list) - len(invalid) - len(techniques)),
    }


print("\nQuery Processor demo:")
demo_query = process_query(
    [
        " t1059.001 ",
        "T1059.001",
        "T1071.001",
        "T1105",
        "T105",
        "not-a-technique",
        "T9999.001",
    ]
)
print(json.dumps(demo_query, indent=2))


## 4. Vector Index

The Vector Index stores one vector per knowledge-base incident.

- **With `GEMINI_API_KEY`:** batch-embed incident text with Gemini (`gemini-embedding-001`, `RETRIEVAL_DOCUMENT`), L2-normalize, store in **FAISS** `IndexFlatIP` (cosine via inner product). Embeddings are cached by content hash so re-runs do not re-call the API.
- **Without a key (or if embedding fails):** skip Gemini and build a **TF-IDF** matrix over group + technique IDs + description, also stored in FAISS when available (else numpy). Retrieval then mixes TF-IDF cosine with **Jaccard** overlap on technique sets.

Incident text used for embeddings / TF-IDF: `group`, `incident_id`, space-separated techniques, and `description`.


In [ ]:
import hashlib
from pathlib import Path

import numpy as np
from typing import Iterable

from sklearn.feature_extraction.text import TfidfVectorizer

try:
    import faiss

    HAS_FAISS = True
except ImportError:
    faiss = None
    HAS_FAISS = False
    print("faiss is not importable; using numpy for similarity search.")

EMBEDDING_MODEL = "gemini-embedding-001"
EMBEDDING_DIM = 768
EMBED_BATCH_SIZE = 32
JACCARD_WEIGHT = 0.7
TFIDF_WEIGHT = 0.3


def incident_to_text(incident: dict) -> str:
    techs = " ".join(incident["techniques"])
    return (
        f"Threat group: {incident['group']}\n"
        f"Incident ID: {incident['incident_id']}\n"
        f"Techniques: {techs}\n"
        f"Description: {incident['description']}"
    )


def query_to_text(techniques: list[str]) -> str:
    return "Observed MITRE ATT&CK techniques: " + " ".join(techniques)


def l2_normalize(matrix: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    norms = np.maximum(norms, 1e-12)
    return (matrix / norms).astype("float32")


def jaccard(a: Iterable[str], b: Iterable[str]) -> float:
    sa, sb = set(a), set(b)
    if not sa and not sb:
        return 1.0
    if not sa or not sb:
        return 0.0
    return len(sa & sb) / len(sa | sb)


def _embed_batches(texts: list[str], task_type: str) -> np.ndarray:
    from google.genai import types

    vectors: list[np.ndarray] = []
    total = len(texts)
    for start in range(0, total, EMBED_BATCH_SIZE):
        batch = texts[start : start + EMBED_BATCH_SIZE]
        result = client.models.embed_content(
            model=EMBEDDING_MODEL,
            contents=batch,
            config=types.EmbedContentConfig(
                task_type=task_type,
                output_dimensionality=EMBEDDING_DIM,
            ),
        )
        for embedding in result.embeddings:
            vectors.append(np.asarray(embedding.values, dtype="float32"))
        done = min(start + EMBED_BATCH_SIZE, total)
        print(f"  Gemini embeddings {done}/{total} ({task_type})")
    return l2_normalize(np.vstack(vectors))


class VectorIndex:
    """FAISS / numpy index over incidents, with a TF-IDF+Jaccard fallback."""

    def __init__(self, incidents: list[dict]):
        self.incidents = incidents
        self.texts = [incident_to_text(inc) for inc in incidents]
        self.backend = "unbuilt"
        self.vectors = None
        self.faiss_index = None
        self.tfidf = None
        self.use_faiss = HAS_FAISS

    def _add_to_store(self, vectors: np.ndarray) -> None:
        self.vectors = np.ascontiguousarray(vectors.astype("float32"))
        if self.use_faiss:
            index = faiss.IndexFlatIP(self.vectors.shape[1])
            index.add(self.vectors)
            self.faiss_index = index
        else:
            self.faiss_index = None

    def _cache_path(self) -> Path:
        digest = hashlib.sha256(EMBEDDING_MODEL.encode())
        for text in self.texts:
            digest.update(b"\0")
            digest.update(text.encode("utf-8"))
        return Path("/tmp") / f"incident_emb_{digest.hexdigest()[:16]}.npz"

    def _build_gemini(self) -> None:
        cache = self._cache_path()
        if cache.is_file():
            loaded = np.load(cache)
            vectors = loaded["vectors"]
            print(f"Loaded cached Gemini embeddings from {cache} shape={vectors.shape}")
        else:
            print(f"Embedding {len(self.texts)} incidents with {EMBEDDING_MODEL} …")
            vectors = _embed_batches(self.texts, task_type="RETRIEVAL_DOCUMENT")
            np.savez_compressed(cache, vectors=vectors)
            print(f"Cached embeddings at {cache}")
        self._add_to_store(vectors)
        self.backend = "gemini"

    def _build_tfidf(self) -> None:
        self.tfidf = TfidfVectorizer(
            lowercase=True,
            token_pattern=r"(?u)\b[a-zA-Z0-9][a-zA-Z0-9\.]+\b",
            ngram_range=(1, 2),
            min_df=1,
            max_features=8000,
        )
        matrix = self.tfidf.fit_transform(self.texts).astype("float32").toarray()
        self._add_to_store(l2_normalize(matrix))
        self.backend = "tfidf_jaccard"
        print(
            f"TF-IDF/Jaccard index ready: {self.vectors.shape[0]} vectors × "
            f"{self.vectors.shape[1]} dims; faiss={bool(self.faiss_index)}"
        )

    def build(self) -> None:
        if HAS_GEMINI and client is not None:
            try:
                self._build_gemini()
                return
            except Exception as exc:
                print(f"SKIP Gemini embeddings ({type(exc).__name__}: {exc}); using TF-IDF/Jaccard.")
        else:
            print("SKIP Gemini embeddings (no API key). Building TF-IDF/Jaccard fallback index.")
        self._build_tfidf()

    def _vector_search(self, query_vec: np.ndarray, k: int) -> tuple[np.ndarray, np.ndarray]:
        query_vec = np.ascontiguousarray(l2_normalize(query_vec.reshape(1, -1)))
        if self.faiss_index is not None:
            scores, ids = self.faiss_index.search(query_vec, k)
            return scores[0], ids[0]
        scores = (self.vectors @ query_vec.T).ravel()
        top = np.argsort(-scores)[:k]
        return scores[top], top

    def search(self, techniques: list[str], k: int = 8) -> list[tuple[int, float]]:
        k = min(k, len(self.incidents))
        if self.backend == "gemini":
            query_vec = _embed_batches([query_to_text(techniques)], task_type="RETRIEVAL_QUERY")
            scores, ids = self._vector_search(query_vec, k)
            return [
                (int(idx), float(score))
                for idx, score in zip(ids, scores)
                if int(idx) >= 0
            ]

        query_vec = self.tfidf.transform([query_to_text(techniques)]).toarray().astype("float32")
        tfidf_scores, _ = self._vector_search(query_vec, len(self.incidents))
        # Recompute full cosine against all rows so we can mix with Jaccard.
        qn = l2_normalize(query_vec.reshape(1, -1))
        cosine = (self.vectors @ qn.T).ravel()
        jac = np.array(
            [jaccard(techniques, inc["techniques"]) for inc in self.incidents],
            dtype="float32",
        )
        combined = JACCARD_WEIGHT * jac + TFIDF_WEIGHT * cosine
        top = np.argsort(-combined)[:k]
        return [(int(idx), float(combined[idx])) for idx in top]


vector_index = VectorIndex(incidents)
vector_index.build()
print("Vector Index backend:", vector_index.backend)


## 5. RAG Retriever

The retriever asks the Vector Index for the nearest incident IDs, then hydrates **group + techniques + description** from the knowledge base (the UML `Index → RAG → DB → RAG` path).

Returned rows are the context that the Gemini step is allowed to use — the system prompt forbids inventing actors or techniques that are not in this list.


In [ ]:
DEFAULT_TOP_K = 8


class RAGRetriever:
    def __init__(self, index: VectorIndex, knowledge_base: list[dict]):
        self.index = index
        self.knowledge_base = knowledge_base

    def retrieve(self, techniques: list[str], k: int = DEFAULT_TOP_K) -> list[dict]:
        if not techniques:
            return []
        hits = self.index.search(techniques, k=k)
        results = []
        for idx, score in hits:
            incident = self.knowledge_base[idx]
            results.append(
                {
                    "score": round(float(score), 4),
                    "incident_id": incident["incident_id"],
                    "group": incident["group"],
                    "techniques": list(incident["techniques"]),
                    "description": incident["description"],
                    "incident": incident,
                }
            )
        return results


retriever = RAGRetriever(vector_index, incidents)

print("RAG demo on the first examples.json query:")
rag_demo_query = process_query(examples[0]["X"]["techniques"])
rag_demo_hits = retriever.retrieve(rag_demo_query["techniques"], k=5)
for hit in rag_demo_hits:
    print(
        f"  score={hit['score']:.4f}  {hit['incident_id']:20s}  {hit['group']:20s}  "
        f"overlap={len(set(hit['techniques']) & set(rag_demo_query['techniques']))}"
    )


## 6. Gemini API

This component fills `prompts/prompt.json`:

- `system_prompt` — analyst instructions: use **only** retrieved incidents; reconstruct an attack chain only when the evidence supports it
- `user_prompt_template` — placeholders `{{techniques}}` and `{{incidents}}`

Gemini is asked for **JSON only**, matching the `y` object in `data/examples.json`:

```json
{
  "groups": [
    {
      "name": "string",
      "match_percentage": 0,
      "matching_techniques": [],
      "similar_incidents": [
        {
          "incident_id": "string",
          "missing_techniques": [],
          "description": "string"
        }
      ],
      "reasoning": "string"
    }
  ]
}
```

The prompt asks for the **top 3 groups**. `examples.json` itself is a *shape* illustration (toy `INC-001` / APT28 rows); live answers come from `incidents_strict.json`.

**Without an API key this cell skips the remote call** and returns `None` so the Result Generator can build the same JSON locally.


In [ ]:
GENERATION_MODEL = "gemini-2.5-flash"  # @param ["gemini-2.5-flash", "gemini-2.0-flash", "gemini-2.5-pro"]


def format_incidents_for_prompt(retrieved: list[dict]) -> str:
    blocks = []
    for hit in retrieved:
        inc = hit["incident"]
        blocks.append(
            "\n".join(
                [
                    f"- incident_id: {inc['incident_id']}",
                    f"  group: {inc['group']}",
                    f"  retrieval_score: {hit['score']}",
                    f"  techniques: {', '.join(inc['techniques'])}",
                    f"  description: {inc['description']}",
                ]
            )
        )
    return "\n".join(blocks) if blocks else "(no incidents retrieved)"


def build_user_prompt(techniques: list[str], retrieved: list[dict]) -> str:
    template = prompt_spec["user_prompt_template"]
    return (
        template.replace("{{techniques}}", json.dumps(techniques, ensure_ascii=False))
        .replace("{{incidents}}", format_incidents_for_prompt(retrieved))
    )


def generate_with_gemini(techniques: list[str], retrieved: list[dict]) -> str | None:
    """Call Gemini. Returns raw JSON text, or None when skipped / failed."""
    if not HAS_GEMINI or client is None:
        print("SKIP Gemini API: no GEMINI_API_KEY. Result Generator will use the local fallback.")
        return None

    from google.genai import types

    user_prompt = build_user_prompt(techniques, retrieved)
    print(f"Calling {GENERATION_MODEL} with {len(retrieved)} retrieved incidents …")
    try:
        response = client.models.generate_content(
            model=GENERATION_MODEL,
            contents=user_prompt,
            config=types.GenerateContentConfig(
                system_instruction=prompt_spec["system_prompt"],
                response_mime_type="application/json",
                temperature=0.2,
            ),
        )
        text = response.text
        print(f"Gemini returned {len(text)} characters of JSON.")
        return text
    except Exception as exc:
        print(f"SKIP Gemini generate_content ({type(exc).__name__}: {exc}). Using fallback.")
        return None


print("Prompt contract:", prompt_spec["name"], "v" + str(prompt_spec.get("version")))
print("system_prompt (first 240 chars):")
print(prompt_spec["system_prompt"][:240], "...")
print("\nFilled user prompt preview (first example, truncated):")
print(build_user_prompt(rag_demo_query["techniques"], rag_demo_hits)[:600], "...")

gemini_demo_text = generate_with_gemini(rag_demo_query["techniques"], rag_demo_hits)
if gemini_demo_text:
    print(gemini_demo_text[:500], "...")


## 7. Result Generator

Parse the model JSON (strip markdown fences if needed), **validate** it against the `y` shape, and if Gemini was skipped or produced invalid JSON, synthesize the same shape from retrieved incidents:

- `match_percentage` = percent of observed techniques that appear in that group's retrieved evidence
- `matching_techniques` = intersection with the query
- `similar_incidents[].missing_techniques` = techniques in the historical incident **not** in the query
- `reasoning` cites concrete incident IDs (the prompt forbids generic “commonly associated” language)

Then plot **match % per group** and a **matching vs missing** table.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:
    def display(obj, **kwargs):
        print(obj)


def parse_model_json(text: str) -> dict:
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned)
        cleaned = re.sub(r"\s*```$", "", cleaned)
    payload = json.loads(cleaned)
    if not isinstance(payload, dict):
        raise ValueError("model output is not a JSON object")
    return payload


def _as_str_list(value, field: str) -> list[str]:
    if value is None:
        return []
    if not isinstance(value, list) or any(not isinstance(x, str) for x in value):
        raise ValueError(f"{field} must be a list of strings")
    return value


def validate_analysis_payload(payload: dict) -> dict:
    """Require the examples.json `y` shape; keep at most 3 groups."""
    if "groups" not in payload or not isinstance(payload["groups"], list):
        raise ValueError("payload.groups must be a list")
    groups = []
    for raw in payload["groups"][:3]:
        if not isinstance(raw, dict):
            raise ValueError("each group must be an object")
        name = raw.get("name")
        if not isinstance(name, str) or not name.strip():
            raise ValueError("group.name must be a non-empty string")
        pct = raw.get("match_percentage")
        if not isinstance(pct, (int, float)) or isinstance(pct, bool):
            raise ValueError(f"group {name}: match_percentage must be a number")
        pct = float(pct)
        if pct < 0 or pct > 100:
            raise ValueError(f"group {name}: match_percentage {pct} not in [0, 100]")
        incidents_out = []
        for inc in raw.get("similar_incidents") or []:
            if not isinstance(inc, dict):
                raise ValueError("similar_incidents entries must be objects")
            iid = inc.get("incident_id")
            desc = inc.get("description")
            if not isinstance(iid, str) or not isinstance(desc, str):
                raise ValueError("similar_incidents need incident_id and description strings")
            incidents_out.append(
                {
                    "incident_id": iid,
                    "missing_techniques": _as_str_list(
                        inc.get("missing_techniques"), "missing_techniques"
                    ),
                    "description": desc,
                }
            )
        reasoning = raw.get("reasoning", "")
        if not isinstance(reasoning, str):
            raise ValueError(f"group {name}: reasoning must be a string")
        groups.append(
            {
                "name": name,
                "match_percentage": round(pct, 1),
                "matching_techniques": _as_str_list(
                    raw.get("matching_techniques"), "matching_techniques"
                ),
                "similar_incidents": incidents_out,
                "reasoning": reasoning,
            }
        )
    return {"groups": groups}


def fallback_from_retrieval(
    techniques: list[str], retrieved: list[dict], top_n: int = 3
) -> dict:
    query = set(techniques)
    by_group: dict[str, list[dict]] = {}
    for hit in retrieved:
        by_group.setdefault(hit["group"], []).append(hit)

    ranked = []
    for group_name, hits in by_group.items():
        union = set()
        for hit in hits:
            union.update(hit["techniques"])
        matching = sorted(query & union)
        coverage = 100.0 * len(matching) / max(len(query), 1)
        mean_score = sum(hit["score"] for hit in hits) / len(hits)
        match_percentage = round(min(100.0, 0.85 * coverage + 15.0 * max(mean_score, 0.0)), 1)
        similar = []
        all_missing = []
        for hit in hits[:3]:
            missing = sorted(set(hit["techniques"]) - query)
            all_missing.extend(missing)
            similar.append(
                {
                    "incident_id": hit["incident_id"],
                    "missing_techniques": missing,
                    "description": hit["description"],
                }
            )
        ids = ", ".join(hit["incident_id"] for hit in hits[:3])
        missing_preview = ", ".join(sorted(set(all_missing))[:12]) or "none listed"
        if matching:
            chain = " → ".join(matching)
            order_note = (
                f"Observed techniques in this group's retrieved evidence, listed in query order: {chain}. "
                "The fallback does not infer a chronological kill chain beyond that listing."
            )
        else:
            order_note = "No observed techniques overlap this group's retrieved incidents, so no attack sequence is claimed."
        reasoning = (
            f"Gemini was skipped; this payload is synthesized from retrieved incidents only. "
            f"Group {group_name} appears in {ids}. Matching observed techniques: {', '.join(matching) or 'none'}. "
            f"Historical incidents also include techniques missing from the query, for example: {missing_preview}. "
            f"{order_note} Match {match_percentage}% is 85% query-coverage plus a small retrieval-score term."
        )
        ranked.append(
            (
                coverage,
                mean_score,
                {
                    "name": group_name,
                    "match_percentage": match_percentage,
                    "matching_techniques": matching,
                    "similar_incidents": similar,
                    "reasoning": reasoning,
                },
            )
        )
    ranked.sort(key=lambda row: (row[0], row[1]), reverse=True)
    return {"groups": [row[2] for row in ranked[:top_n]]}


def generate_result(
    techniques: list[str], retrieved: list[dict], gemini_text: str | None
) -> tuple[dict, str]:
    if gemini_text:
        try:
            parsed = validate_analysis_payload(parse_model_json(gemini_text))
            return parsed, "gemini"
        except Exception as exc:
            print(f"Gemini JSON failed validation ({exc}); using retrieval fallback.")
    payload = validate_analysis_payload(fallback_from_retrieval(techniques, retrieved))
    return payload, "tfidf_jaccard_fallback"


def shorten_ids(values: list[str], limit: int = 10) -> str:
    if not values:
        return "—"
    if len(values) <= limit:
        return ", ".join(values)
    return ", ".join(values[:limit]) + f" … (+{len(values) - limit} more)"


def visualize_result(payload: dict, title: str = "Threat group match %") -> pd.DataFrame:
    groups = payload.get("groups") or []
    if not groups:
        print("No groups to visualize.")
        return pd.DataFrame()

    fig, ax = plt.subplots(figsize=(8, 4))
    names = [g["name"] for g in groups]
    pcts = [g["match_percentage"] for g in groups]
    ax.bar(names, pcts, color="#3b6d9a")
    ax.set_ylim(0, 100)
    ax.set_ylabel("match_percentage")
    ax.set_title(title)
    ax.axhline(50, color="#bbbbbb", linewidth=0.8, linestyle="--")
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()
    if IN_COLAB:
        plt.show()
    else:
        plt.close(fig)

    rows = []
    for g in groups:
        missing = []
        seen_m = set()
        for inc in g["similar_incidents"]:
            for tid in inc["missing_techniques"]:
                if tid not in seen_m:
                    seen_m.add(tid)
                    missing.append(tid)
        rows.append(
            {
                "group": g["name"],
                "match_percentage": g["match_percentage"],
                "matching_techniques": shorten_ids(g["matching_techniques"]),
                "missing_techniques": shorten_ids(missing, limit=8),
                "similar_incident_ids": ", ".join(
                    inc["incident_id"] for inc in g["similar_incidents"]
                )
                or "—",
            }
        )
    table = pd.DataFrame(rows)
    display(table)
    return table


demo_payload, demo_backend = generate_result(
    rag_demo_query["techniques"], rag_demo_hits, gemini_demo_text
)
print("Result Generator backend:", demo_backend)
print(json.dumps(demo_payload, indent=2)[:1200], "...")
visualize_result(demo_payload, title="Demo (first examples.json query)")


## 8. REST API — `POST /analyze`

`analyze(techniques)` is the function the UML REST API would call. It wires Query Processor → RAG Retriever → Gemini API (or skip) → Result Generator and returns JSON with a `groups` array in the `y` shape, plus diagnostics (`backend`, invalid/unknown IDs).

The optional FastAPI app below is started in a **background thread** (Colab-friendly) and called once with `requests` to show the HTTP contract:

```http
POST /analyze
Content-Type: application/json

{"techniques": ["T1059.001", "T1071.001", "T1105"]}
```


In [ ]:
def analyze(techniques, k: int = DEFAULT_TOP_K, top_groups: int = 3) -> dict:
    """End-to-end Threat Actor Analysis API (the POST /analyze handler)."""
    query = process_query(techniques)
    retrieved = retriever.retrieve(query["techniques"], k=k)
    gemini_text = generate_with_gemini(query["techniques"], retrieved)
    payload, backend = generate_result(query["techniques"], retrieved, gemini_text)
    payload["groups"] = payload["groups"][:top_groups]
    payload["validated_techniques"] = query["techniques"]
    payload["invalid_techniques"] = query["invalid"]
    payload["unknown_techniques"] = query["unknown"]
    payload["backend"] = backend
    payload["retrieved_incident_ids"] = [hit["incident_id"] for hit in retrieved]
    return payload


api_demo = analyze(examples[0]["X"]["techniques"])
print("analyze() keys:", sorted(api_demo.keys()))
print("backend:", api_demo["backend"], "groups:", [g["name"] for g in api_demo["groups"]])


Optional in-process FastAPI demo. If the port is busy or FastAPI is missing, the `analyze()` function above is still the API.


In [ ]:
import threading
import time

API_HOST = "127.0.0.1"
API_PORT = 8765
FASTAPI_STARTED = globals().get("FASTAPI_STARTED", False)

try:
    from fastapi import FastAPI
    from pydantic import BaseModel
    import uvicorn
    import requests

    class AnalyzeRequest(BaseModel):
        techniques: list[str]

    if "app" not in globals():
        app = FastAPI(title="Threat Actor Analysis API", version="0.1.0")

        @app.post("/analyze")
        def analyze_endpoint(body: AnalyzeRequest):
            return analyze(body.techniques)

    def _serve():
        config = uvicorn.Config(app, host=API_HOST, port=API_PORT, log_level="warning")
        server = uvicorn.Server(config)
        server.install_signal_handlers = lambda: None
        server.run()

    if not FASTAPI_STARTED:
        threading.Thread(target=_serve, daemon=True).start()
        for _ in range(30):
            try:
                requests.get(f"http://{API_HOST}:{API_PORT}/openapi.json", timeout=0.3)
                break
            except Exception:
                time.sleep(0.2)
        FASTAPI_STARTED = True
        print(f"FastAPI listening on http://{API_HOST}:{API_PORT}")

    http_response = requests.post(
        f"http://{API_HOST}:{API_PORT}/analyze",
        json={"techniques": examples[0]["X"]["techniques"]},
        timeout=120,
    )
    print("POST /analyze status:", http_response.status_code)
    body = http_response.json()
    print("HTTP backend:", body.get("backend"), "groups:", [g["name"] for g in body.get("groups", [])])
except Exception as exc:
    print(f"FastAPI demo skipped ({type(exc).__name__}: {exc}). analyze() still works.")


## 9. End-to-end run of `data/examples.json`

Each example has `X.techniques` (the API input) and `y` (the **required output shape**, not a gold label for `incidents_strict.json`). We run `analyze(X.techniques)` for all three queries, print the JSON, and reuse the Result Generator charts/tables.

Path: **input technique IDs → Query Processor → Vector Index + RAG → Gemini or fallback → validated `groups` JSON → chart + table**.


In [ ]:
from pathlib import Path

ARTIFACT_DIR = Path("/opt/cursor/artifacts")
if not ARTIFACT_DIR.is_dir():
    ARTIFACT_DIR = Path("/tmp")

e2e_results = []
for i, example in enumerate(examples, start=1):
    x_techs = example["X"]["techniques"]
    print("\n" + "=" * 72)
    print(f"Example {i}/{len(examples)}  X.techniques = {x_techs}")
    print("y shape groups (schema illustration only):", [g["name"] for g in example["y"]["groups"]])
    result = analyze(x_techs)
    e2e_results.append(result)
    print("backend:", result["backend"])
    print("validated:", result["validated_techniques"])
    print("retrieved:", result["retrieved_incident_ids"])
    print(json.dumps({"groups": result["groups"]}, indent=2)[:2000])
    table = visualize_result(
        result, title=f"Example {i}: match %  ({result['backend']})"
    )
    fig_path = ARTIFACT_DIR / f"example_{i}_match_percent.png"
    # Re-save the last figure if matplotlib still has it; otherwise draw again.
    if result["groups"]:
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.bar(
            [g["name"] for g in result["groups"]],
            [g["match_percentage"] for g in result["groups"]],
            color="#3b6d9a",
        )
        ax.set_ylim(0, 100)
        ax.set_ylabel("match_percentage")
        ax.set_title(f"Example {i}: match % ({result['backend']})")
        plt.xticks(rotation=25, ha="right")
        plt.tight_layout()
        fig.savefig(fig_path, dpi=120)
        plt.close(fig)
        print("saved", fig_path)

print("\nAll", len(e2e_results), "examples finished. Backends:", [r["backend"] for r in e2e_results])


### Notes for grading / reuse

- **With a key:** Vector Index uses Gemini embeddings; Gemini API fills `reasoning` from retrieved incidents only (`prompts/prompt.json` v1.1).
- **Without a key:** those two steps print `SKIP` and the TF-IDF/Jaccard path still returns valid `y`-shaped JSON.
- To overwrite `data/examples.json` with **real** Gemini outputs (never placeholders), run `python scripts/regenerate_examples.py` locally with `GEMINI_API_KEY` set.
- Dataset copyright: © 2026 The MITRE Corporation. See the repository README.
